# Navigation Route Finder · Aachen

A small, self-built navigation system: type in two addresses, get the shortest driving route through the real street network of Aachen, drawn on an interactive map.

**Pipeline**

1. **Street network** of Aachen from OpenStreetMap (OSMnx) → a directed graph: intersections = nodes, street segments = edges with their length in metres
2. **Geocoding**: address → latitude / longitude (Nominatim)
3. **Snapping**: coordinates → nearest intersection in the graph
4. **Dijkstra**: shortest path between the two intersections, weighted by length
5. **Map**: route, start / end markers and base map styles (Folium)

> Originally a Streamlit app. This notebook is the cleaned-up version of the same idea, with Dijkstra implemented by hand and checked against NetworkX.

## 1 · Setup

In [3]:
# Run once (in Colab or a fresh environment)
%pip install osmnx networkx geopy folium matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.7/104.7 kB 857.7 kB/s eta 0:00:00


In [4]:
import heapq
import time
from pathlib import Path

import folium
import networkx as nx
import osmnx as ox
from geopy.geocoders import Nominatim

In [5]:
############################ CONFIG #################################
PLACE_NAME   = "Aachen, Germany"
NETWORK_TYPE = "drive"                              # 'drive', 'walk' or 'bike'
CACHE_FILE   = Path("aachen_drive.graphml")         # avoids downloading the network every run

START_ADDRESS = "Templergraben 55, Aachen"          # RWTH main building
END_ADDRESS   = "Aachen Hauptbahnhof"

## 2 · Street network

OSMnx downloads all drivable streets inside the city boundary and turns them into a `MultiDiGraph`:
- **directed**, so one-way streets are respected
- **multi**, so two intersections can be linked by more than one street
- every edge carries its `length` in metres, which is the weight Dijkstra minimises later

The first download takes a while, so the graph is cached as a GraphML file.

In [6]:
def get_graph(place_name, network_type, cache_file=None):
    # Load the street network from cache if available, otherwise download and cache it
    if cache_file is not None and Path(cache_file).exists():
        return ox.load_graphml(cache_file)
    graph = ox.graph_from_place(place_name, network_type=network_type)
    if cache_file is not None:
        ox.save_graphml(graph, cache_file)
    return graph


start_time = time.time()
graph = get_graph(PLACE_NAME, NETWORK_TYPE, CACHE_FILE)
print(f"{graph.number_of_nodes():,} intersections, {graph.number_of_edges():,} street segments "
      f"({time.time() - start_time:.1f} s)")

3,861 intersections, 9,103 street segments (111.6 s)


## 3 · From address to graph node

Two steps: Nominatim turns the address into coordinates, then OSMnx finds the closest intersection in the graph. The route always starts and ends at an intersection, not at the exact front door.

In [7]:
geolocator = Nominatim(user_agent="navigation_route_finder")   # Nominatim requires a user agent


def get_nearest_node(graph, address):
    # Convert an address to the nearest graph node; returns None if the address is not found
    location = geolocator.geocode(address)
    if location is None:
        print(f"Address '{address}' not found.")
        return None
    return ox.distance.nearest_nodes(graph, X=location.longitude, Y=location.latitude)


start_node = get_nearest_node(graph, START_ADDRESS)
end_node   = get_nearest_node(graph, END_ADDRESS)
print("Start node:", start_node, "| End node:", end_node)

Start node: 27290892 | End node: 27293790


## 4 · Dijkstra's algorithm

Idea: explore the network outwards from the start like a growing wave, always continuing from the **closest not-yet-finished intersection**. Once an intersection is taken from the queue, its distance is final. When the destination is taken from the queue, the shortest route is found.

| Step | What happens |
|---|---|
| Start | distance(start) = 0, everything else = ∞; put start in a priority queue |
| Loop | take the node `u` with the smallest distance from the queue |
| Relax | for every neighbour `v`: if `dist[u] + length(u, v) < dist[v]`, update `dist[v]` and remember `u` as its predecessor |
| Stop | when the destination is taken from the queue, walk back through the predecessors |

The priority queue (`heapq`) makes the "closest next node" lookup fast.

In [8]:
def edge_length(graph, u, v):
    # In a MultiDiGraph several streets can connect u and v -> use the shortest one
    return min(data["length"] for data in graph[u][v].values())


def dijkstra(graph, source, target):
    # Own implementation of Dijkstra's shortest path, weight = edge length in metres
    dist = {source: 0.0}          # best known distance from the source
    prev = {}                     # predecessor of each node on its best path
    done = set()                  # nodes whose distance is final
    queue = [(0.0, source)]       # priority queue of (distance, node)

    while queue:
        d, u = heapq.heappop(queue)
        if u in done:             # outdated queue entry, a shorter one was handled already
            continue
        done.add(u)
        if u == target:
            break
        for v in graph.successors(u):
            new_d = d + edge_length(graph, u, v)
            if new_d < dist.get(v, float("inf")):
                dist[v] = new_d
                prev[v] = u
                heapq.heappush(queue, (new_d, v))

    if target not in done:
        return None, None, len(done)          # destination not reachable

    # Walk back from the target to rebuild the route
    route = [target]
    while route[-1] != source:
        route.append(prev[route[-1]])
    return route[::-1], dist[target], len(done)

In [9]:
def get_shortest_route(graph, start_node, end_node):
    # Shortest route with the own Dijkstra, cross-checked against NetworkX
    route, route_length, n_settled = dijkstra(graph, start_node, end_node)
    if route is None:
        return None, None

    nx_length = nx.shortest_path_length(graph, start_node, end_node, weight="length")
    assert abs(nx_length - route_length) < 1e-6, "own Dijkstra disagrees with NetworkX"

    print(f"Route: {len(route)} intersections, {route_length / 1000:.2f} km")
    print(f"Dijkstra settled {n_settled:,} of {graph.number_of_nodes():,} intersections before reaching the destination")
    return route, route_length


route, route_length = get_shortest_route(graph, start_node, end_node)

Route: 22 intersections, 2.22 km
Dijkstra settled 497 of 3,861 intersections before reaching the destination


## 5 · Route on an interactive map

The route is drawn along the real street geometry (OSMnx stores the curve of each street segment), with markers for start and destination and a choice of base maps in the layer control.

In [12]:
# Base maps (all free, no API key needed)
ESRI = "https://server.arcgisonline.com/ArcGIS/rest/services"
LIGHT_URL = ESRI + "/Canvas/World_Light_Gray_Base/MapServer/tile/{z}/{y}/{x}"
SATELLITE_URL = ESRI + "/World_Imagery/MapServer/tile/{z}/{y}/{x}"

TILES = {
    "Light": dict(tiles=LIGHT_URL, attr="Tiles © Esri"),
    "OpenStreetMap": dict(tiles="OpenStreetMap",
                          attr="© OpenStreetMap contributors"),
    "Satellite": dict(tiles=SATELLITE_URL, attr="Tiles © Esri"),
}


def route_coordinates(graph, route):
    # Follow the street geometry of every edge on the route;
    # fall back to straight lines
    coords = []
    for u, v in zip(route[:-1], route[1:]):
        data = min(graph[u][v].values(), key=lambda d: d["length"])
        if "geometry" in data:
            points = [(lat, lon) for lon, lat in data["geometry"].coords]
        else:
            points = [(graph.nodes[u]["y"], graph.nodes[u]["x"]),
                      (graph.nodes[v]["y"], graph.nodes[v]["x"])]
        coords.extend(points if not coords else points[1:])
    return coords


def plot_route_on_map(graph, route, start_label, end_label,
                      map_style="Light", show_network=False):
    coords = route_coordinates(graph, route)

    m = folium.Map(location=coords[len(coords) // 2],
                   zoom_start=14, tiles=None)

    # All base maps available, the chosen one is shown first
    for name, tile in TILES.items():
        folium.TileLayer(**tile, name=name,
                         show=(name == map_style)).add_to(m)

    # Optional: whole street network (slow for large cities)
    if show_network:
        edges = ox.graph_to_gdfs(graph, nodes=False)
        folium.GeoJson(edges.__geo_interface__, name="Street network",
                       style_function=lambda x: {"color": "#999999",
                                                 "weight": 1}).add_to(m)

    # Route + markers as one toggleable layer
    route_layer = folium.FeatureGroup(name="Route")
    folium.PolyLine(coords, color="red", weight=5,
                    opacity=0.8).add_to(route_layer)
    folium.Marker(coords[0], popup=f"Start: {start_label}",
                  icon=folium.Icon(color="green")).add_to(route_layer)
    folium.Marker(coords[-1], popup=f"End: {end_label}",
                  icon=folium.Icon(color="red")).add_to(route_layer)
    route_layer.add_to(m)

    folium.LayerControl().add_to(m)

    # Zoom the map to the route
    lats = [p[0] for p in coords]
    lons = [p[1] for p in coords]
    m.fit_bounds([[min(lats), min(lons)], [max(lats), max(lons)]])
    return m


if route:
    route_map = plot_route_on_map(graph, route, START_ADDRESS,
                                  END_ADDRESS, map_style="Light")
else:
    route_map = None
    print("No route found.")
route_map

In [ ]:
# Optional: save the map as a standalone HTML file
if route_map is not None:
    route_map.save("route_map.html")

## 6 · Limitations and ideas

- **Shortest ≠ fastest.** The weight is length only. With `ox.add_edge_speeds` and `ox.add_edge_travel_times`, the same Dijkstra can minimise travel time instead.
- **Intersections, not doors.** Start and end are snapped to the nearest intersection, so the last metres to the address are missing.
- **Geocoding can be ambiguous.** Nominatim returns its best match; add the city to every address and respect its usage limit (about one request per second).
- **City boundary.** Routes leaving Aachen are not possible, because only the city's network is loaded.
- **Speed-up.** A* adds a straight-line-distance estimate to Dijkstra and settles far fewer intersections on long routes.